<a href="https://colab.research.google.com/github/hmdaalln/Final_dataset/blob/main/Web_Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Web Application Development

In this section, the selected DnCNN model is prepared for deployment in a Flask web application.

The application will allow a user to upload a noisy image, process it using the trained DnCNN model, and display the denoised result.

The web application files are organized separately from the training code so that the final model can later be packaged with Docker and deployed.

# 1. Create Project Folders

In [ ]:
import os

# Main folder for the Flask web application
project_root = "/content/image_denoising_app"

# Create the required project folders
folders = [
    project_root,
    os.path.join(project_root, "models"),
    os.path.join(project_root, "templates"),
    os.path.join(project_root, "static"),
    os.path.join(project_root, "static", "uploads"),
    os.path.join(project_root, "static", "results")
]

for folder in folders:
    os.makedirs(
        folder,
        exist_ok=True
    )

print(
    "Web application folders "
    "created successfully!"
)

**Why this code?**

This code creates the folder structure required for the Flask web application.

The models folder will store the trained DnCNN weights, the templates folder will contain the HTML interface, and the static folder will store uploaded images and denoised results.

Organizing the files into separate folders makes the application easier to manage and prepares it for Docker packaging and deployment.


## 2. Display Folder Strcture

In [ ]:
# Display the project folder structure

for root, dirs, files in os.walk(
    project_root
):

    level = root.replace(
        project_root,
        ""
    ).count(os.sep)

    indent = "    " * level

    print(
        f"{indent}"
        f"{os.path.basename(root)}/"
    )

    for file in files:

        print(
            f"{indent}    {file}"
        )

**Why this code?**

This code displays the folder structure so that we can verify that the required web-application directories were created correctly before adding the model and application files.


## 3. Get the Trained DnCNN Model

In [ ]:
import os
import shutil

# Path to the selected trained DnCNN model
source_model_path = (
    "/content/drive/MyDrive/cap/"
    "ISOCELL_GN1_Results/checkpoints/"
    "dncnn_final_weights.pth"
)

# Destination inside the web application
destination_model_path = os.path.join(
    project_root,
    "models",
    "dncnn_final_weights.pth"
)

# Check that the trained model exists
print(
    "Source model exists:",
    os.path.isfile(source_model_path)
)

if not os.path.isfile(source_model_path):
    raise FileNotFoundError(
        f"Model not found: {source_model_path}"
    )

# Copy the model
shutil.copy2(
    source_model_path,
    destination_model_path
)

print(
    "Model copied successfully!"
)

print(
    "Destination:",
    destination_model_path
)

print(
    "Model exists in web app:",
    os.path.isfile(destination_model_path)
)

**Why this code?**

This code copies the selected trained DnCNN weights into the web-application project. Keeping the model inside the application folder makes it easier to package and deploy the complete application later.


## 4. Test the Trained Model

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt


class DnCNN(nn.Module):

    def __init__(
        self,
        image_channels=3,
        depth=17,
        features=128
    ):
        super().__init__()

        layers = []

        layers.append(
            nn.Conv2d(
                image_channels,
                features,
                kernel_size=3,
                padding=1,
                bias=True
            )
        )

        layers.append(
            nn.ReLU(inplace=True)
        )

        for _ in range(depth - 2):

            layers.append(
                nn.Conv2d(
                    features,
                    features,
                    kernel_size=3,
                    padding=1,
                    bias=False
                )
            )

            layers.append(
                nn.BatchNorm2d(features)
            )

            layers.append(
                nn.ReLU(inplace=True)
            )

        layers.append(
            nn.Conv2d(
                features,
                image_channels,
                kernel_size=3,
                padding=1,
                bias=True
            )
        )

        self.model = nn.Sequential(*layers)

    def forward(self, x):

        return self.model(x)

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

app_model = DnCNN(
    image_channels=3,
    depth=17,
    features=128
).to(device)

checkpoint = torch.load(
    destination_model_path,
    map_location=device
)

app_model.load_state_dict(
    checkpoint["model_state_dict"]
)

app_model.eval()

print("Trained DnCNN model loaded successfully!")
print("Device:", device)

In [ ]:
# Use one existing noisy test image
test_image_path = test_pairs[0][
    "noisy_path"
]

test_image = Image.open(
    test_image_path
).convert("RGB")

input_tensor = TF.to_tensor(
    test_image
).unsqueeze(0).to(device)

with torch.no_grad():

    predicted_noise = app_model(
        input_tensor
    )

    denoised_tensor = torch.clamp(
        input_tensor
        - predicted_noise,
        0,
        1
    )

denoised_image = TF.to_pil_image(
    denoised_tensor
    .squeeze(0)
    .cpu()
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(test_image)
plt.title("Noisy Input")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(denoised_image)
plt.title("DnCNN Output")
plt.axis("off")

plt.tight_layout()
plt.show()

**Why this code?**

This step verifies that the saved DnCNN model can be loaded independently from the training process and used for image inference before integrating it into Flask.



## 5. Create Flask Application - app.py

In [ ]:
app_code = r'''
import os
import time
import uuid

import torch
import torch.nn as nn

from flask import (
    Flask,
    render_template,
    request
)

from PIL import Image
from torchvision.transforms import functional as TF


class DnCNN(nn.Module):

    def __init__(
        self,
        image_channels=3,
        depth=17,
        features=128
    ):
        super().__init__()

        layers = []

        layers.append(
            nn.Conv2d(
                image_channels,
                features,
                kernel_size=3,
                padding=1,
                bias=True
            )
        )

        layers.append(
            nn.ReLU(inplace=True)
        )

        for _ in range(depth - 2):

            layers.append(
                nn.Conv2d(
                    features,
                    features,
                    kernel_size=3,
                    padding=1,
                    bias=False
                )
            )

            layers.append(
                nn.BatchNorm2d(features)
            )

            layers.append(
                nn.ReLU(inplace=True)
            )

        layers.append(
            nn.Conv2d(
                features,
                image_channels,
                kernel_size=3,
                padding=1,
                bias=True
            )
        )

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


BASE_DIR = os.path.dirname(
    os.path.abspath(__file__)
)

UPLOAD_FOLDER = os.path.join(
    BASE_DIR,
    "static",
    "uploads"
)

RESULT_FOLDER = os.path.join(
    BASE_DIR,
    "static",
    "results"
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "dncnn_final_weights.pth"
)

os.makedirs(
    UPLOAD_FOLDER,
    exist_ok=True
)

os.makedirs(
    RESULT_FOLDER,
    exist_ok=True
)


app = Flask(__name__)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


def load_dncnn_model():

    model = DnCNN(
        image_channels=3,
        depth=17,
        features=128
    ).to(device)

    checkpoint = torch.load(
        MODEL_PATH,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    return model


model = load_dncnn_model()


def denoise_image(
    input_path,
    output_path
):

    image = Image.open(
        input_path
    ).convert("RGB")

    noisy_tensor = TF.to_tensor(
        image
    ).unsqueeze(0).to(device)

    start_time = time.perf_counter()

    with torch.no_grad():

        predicted_noise = model(
            noisy_tensor
        )

        denoised_tensor = torch.clamp(
            noisy_tensor
            - predicted_noise,
            0,
            1
        )

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time = (
        time.perf_counter()
        - start_time
    )

    denoised_image = TF.to_pil_image(
        denoised_tensor
        .squeeze(0)
        .cpu()
    )

    denoised_image.save(
        output_path
    )

    return inference_time


@app.route(
    "/",
    methods=["GET", "POST"]
)
def index():

    original_image = None
    denoised_image = None
    inference_time = None
    error_message = None

    if request.method == "POST":

        uploaded_file = request.files.get(
            "image"
        )

        if (
            uploaded_file is None
            or uploaded_file.filename == ""
        ):

            error_message = (
                "Please select an image."
            )

        else:

            try:

                extension = os.path.splitext(
                    uploaded_file.filename
                )[1].lower()

                if extension not in [
                    ".png",
                    ".jpg",
                    ".jpeg"
                ]:
                    raise ValueError(
                        "Only PNG, JPG, and JPEG "
                        "images are supported."
                    )

                unique_name = (
                    str(uuid.uuid4())
                    + extension
                )

                upload_path = os.path.join(
                    UPLOAD_FOLDER,
                    unique_name
                )

                result_name = (
                    "denoised_"
                    + unique_name
                )

                result_path = os.path.join(
                    RESULT_FOLDER,
                    result_name
                )

                uploaded_file.save(
                    upload_path
                )

                inference_time = denoise_image(
                    upload_path,
                    result_path
                )

                original_image = (
                    "uploads/"
                    + unique_name
                )

                denoised_image = (
                    "results/"
                    + result_name
                )

            except Exception as error:

                error_message = str(
                    error
                )

    return render_template(
        "index.html",
        original_image=original_image,
        denoised_image=denoised_image,
        inference_time=inference_time,
        error_message=error_message
    )


if __name__ == "__main__":

    app.run(
        host="0.0.0.0",
        port=5000
    )
'''

app_path = os.path.join(
    project_root,
    "app.py"
)

with open(
    app_path,
    "w"
) as file:

    file.write(
        app_code
    )

print(
    "app.py created successfully!"
)

## 6. Confirm app.py Was Created

In [ ]:
app_path = os.path.join(
    project_root,
    "app.py"
)

print(
    "app.py exists:",
    os.path.isfile(app_path)
)

print(
    "app.py size:",
    os.path.getsize(app_path),
    "bytes"
)

**Why this code?**

This step confirms that the Flask backend file was created successfully before creating the user interface.

## 7. Create index.html

In [ ]:
html_code = r'''
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="UTF-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <title>
        DnCNN Image Denoising
    </title>

    <style>

        body {
            font-family: Arial, sans-serif;
            background: #f4f6f8;
            margin: 0;
            padding: 0;
        }

        .container {
            width: 90%;
            max-width: 1100px;
            margin: 40px auto;
            background: white;
            padding: 30px;
            border-radius: 12px;
            box-shadow:
                0 4px 15px
                rgba(0, 0, 0, 0.08);
        }

        h1 {
            text-align: center;
            margin-bottom: 10px;
        }

        .description {
            text-align: center;
            margin-bottom: 30px;
        }

        .upload-form {
            text-align: center;
            margin-bottom: 30px;
        }

        input[type="file"] {
            margin-bottom: 15px;
        }

        button {
            padding: 10px 24px;
            border: none;
            border-radius: 6px;
            cursor: pointer;
            font-size: 16px;
        }

        .comparison {
            display: flex;
            gap: 20px;
            margin-top: 30px;
        }

        .image-box {
            flex: 1;
            text-align: center;
        }

        .image-box img {
            max-width: 100%;
            border-radius: 8px;
            border: 1px solid #ddd;
        }

        .metric {
            text-align: center;
            margin-top: 20px;
            font-weight: bold;
        }

        .error {
            text-align: center;
            margin: 20px;
        }

        @media (
            max-width: 768px
        ) {

            .comparison {
                flex-direction: column;
            }
        }

    </style>

</head>


<body>

<div class="container">

    <h1>
        Deep Learning Image Denoising
    </h1>

    <p class="description">
        Upload a noisy image and use the trained
        DnCNN model to generate a denoised result.
    </p>


    <form
        class="upload-form"
        method="POST"
        enctype="multipart/form-data"
    >

        <input
            type="file"
            name="image"
            accept=".png,.jpg,.jpeg"
            required
        >

        <br>

        <button type="submit">
            Denoise Image
        </button>

    </form>


    {% if error_message %}

        <p class="error">
            {{ error_message }}
        </p>

    {% endif %}


    {% if original_image and denoised_image %}

    <div class="comparison">

        <div class="image-box">

            <h3>
                Noisy Input
            </h3>

            <img
                src="{{ url_for(
                    'static',
                    filename=original_image
                ) }}"
            >

        </div>


        <div class="image-box">

            <h3>
                DnCNN Output
            </h3>

            <img
                src="{{ url_for(
                    'static',
                    filename=denoised_image
                ) }}"
            >

        </div>

    </div>


    <p class="metric">

        Inference Time:
        {{ "%.4f"|format(inference_time) }}
        seconds

    </p>

    {% endif %}

</div>

</body>

</html>
'''

template_path = os.path.join(
    project_root,
    "templates",
    "index.html"
)

with open(
    template_path,
    "w"
) as file:

    file.write(
        html_code
    )

print("index.html created successfully!")

**Why this code?**

This HTML file creates the user interface for the Flask application. It allows the user to upload an image and displays the noisy input, DnCNN output, and inference time

## 8. Create requirements.txt - Update

In [ ]:
requirements = """flask
torch
torchvision
pillow
numpy
gunicorn
"""

requirements_path = os.path.join(
    project_root,
    "requirements.txt"
)

with open(
    requirements_path,
    "w"
) as file:

    file.write(
        requirements
    )

print(
    "requirements.txt "
    "created successfully!"
)

## 9. Create Dockerfile

In [ ]:
dockerfile_code = r'''
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install \
    --no-cache-dir \
    -r requirements.txt

COPY . .

EXPOSE 5000

CMD [
    "gunicorn",
    "--bind",
    "0.0.0.0:5000",
    "--timeout",
    "120",
    "app:app"
]
'''

dockerfile_path = os.path.join(
    project_root,
    "Dockerfile"
)

with open(
    dockerfile_path,
    "w"
) as file:

    file.write(
        dockerfile_code
    )

print("Dockerfile created successfully!")

##10. Create .gitignore

In [ ]:
gitignore_code = r'''
__pycache__/
*.pyc
.DS_Store

static/uploads/*
static/results/*
'''

gitignore_path = os.path.join(
    project_root,
    ".gitignore"
)

with open(
    gitignore_path,
    "w"
) as file:

    file.write(
        gitignore_code
    )

print(".gitignore created successfully!")

## 11. Verify Project Files

In [ ]:
required_files = [
    "app.py",
    "requirements.txt",
    "Dockerfile",
    os.path.join(
        "models",
        "dncnn_final_weights.pth"
    ),
    os.path.join(
        "templates",
        "index.html"
    )
]

print(
    "Image denoising"
)

for item in required_files:

    full_path = os.path.join(
        project_root,
        item
    )

    print(
        item,
        "->",
        os.path.exists(full_path)
    )

In [ ]:
for root, dirs, files in os.walk(
    project_root
):

    level = root.replace(
        project_root,
        ""
    ).count(os.sep)

    indent = "    " * level

    print(
        f"{indent}"
        f"{os.path.basename(root)}/"
    )

    for file in files:

        print(
            f"{indent}    {file}"
        )

## 12. Test the Flask Application

In [ ]:
!pip install flask gunicorn -q

In [ ]:
import sys

sys.path.insert(
    0,
    project_root
)

from app import app

print(
    "Flask application "
    "imported successfully!"
)

print(
    "Registered routes:"
)

print(
    app.url_map
)

## 13. Run Flask in Colab

In [ ]:
%cd /content/image_denoising_app
!python app.py

## 14. Docker Test

In [ ]:
docker build -t dncnn-image-denoising .

In [ ]:
docker run -p 5000:5000 dncnn-image-denoising

## 15. Push the Application to GitHub

In [ ]:
app.py
Dockerfile
requirements.txt
.gitignore
models/dncnn_final_weights.pth
templates/index.html
static/

## 16. Deploy the Web Application

In [ ]:
app.py
Dockerfile
requirements.txt
.gitignore
models/dncnn_final_weights.pth
templates/index.html
static/

**Why this step?**

Deployment makes the trained DnCNN model available through a public web application. The deployment platform builds the Docker image from the GitHub repository and starts the Flask application using Gunicorn.
